In [2]:
import pandas as pd

df_clean = pd.read_csv(
    r"I:\Projects\iocl\data\df_clean.csv"
)

df_clean["DATE"] = pd.to_datetime(df_clean["DATE"])

In [3]:
# ── Baseline RandomForestRegressor ──────────────────────────────────────────

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Define features and target
X = df_clean.drop(columns=['Wthr', 'DATE'])
y = df_clean['Wthr']

# 2. Split into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Features used    : {X_train.shape[1]}\n")

# 3. Train the model
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# 4. Make predictions
y_pred = model.predict(X_test)

# 5. Evaluate
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("── Model Performance ───────────────────────────────")
print(f"  MAE  (Mean Absolute Error)       : {mae:.4f}")
print(f"  RMSE (Root Mean Squared Error)   : {rmse:.4f}")
print(f"  R²   (Coefficient of Det.)       : {r2:.4f}")
print("────────────────────────────────────────────────────\n")

# 6. First 10 actual vs predicted
comparison = pd.DataFrame({
    'Actual'   : y_test.values[:10],
    'Predicted': y_pred[:10].round(4),
    'Error'    : (y_test.values[:10] - y_pred[:10]).round(4)
})
print("── First 10 Actual vs Predicted ────────────────────")
print(comparison.to_string(index=False))
print()

# 7. Top 10 feature importances
importances = pd.Series(model.feature_importances_, index=X.columns)
top10 = importances.sort_values(ascending=False).head(10)

print("── Top 10 Feature Importances ──────────────────────")
for feat, score in top10.items():
    bar = '█' * int(score * 100)
    print(f"  {feat:<25} {score:.4f}  {bar}")
print("────────────────────────────────────────────────────")

Training samples : 366
Test samples     : 92
Features used    : 22

── Model Performance ───────────────────────────────
  MAE  (Mean Absolute Error)       : 0.9190
  RMSE (Root Mean Squared Error)   : 1.1618
  R²   (Coefficient of Det.)       : -0.0122
────────────────────────────────────────────────────

── First 10 Actual vs Predicted ────────────────────
 Actual  Predicted  Error
   -2.0     -1.200 -0.800
   -1.0     -0.756 -0.244
   -1.0     -1.556  0.556
   -3.0     -1.584 -1.416
   -2.0     -0.896 -1.104
   -2.0     -1.264 -0.736
    1.0     -0.852  1.852
   -1.0     -0.726 -0.274
    1.1     -1.236  2.336
   -1.0     -1.164  0.164

── Top 10 Feature Importances ──────────────────────
  Stabilizer Reflux Flow    0.0824  ████████
  LPG Flow                  0.0760  ███████
  HGO CR to reboiler Flow   0.0747  ███████
  Stabiliser bottom level   0.0699  ██████
  HGO CR Reboiler Inlet Temp( TI-1914) 0.0618  ██████
  Stabilizer Feed T         0.0591  █████
  HGO CR Flow              

# Experiments
Experiment 1 Hypothesis - Off Spec 32 is ruining stuffs
Result - No Conclusion, instead removing it completely made it worse 

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# =====================================================
# Experiment 1 : Remove Faulty Sensor
# =====================================================

# Create experimental dataset
df_exp = df_clean.drop(columns=["Off Spec LPG flow (tag Faulty)"]).copy()

# Features and Target
X = df_exp.drop(columns=["DATE", "Wthr"])
y = df_exp["Wthr"]

# Train-Test Split (same settings as before)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Train Model (same settings as baseline)
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Training samples :", len(X_train))
print("Test samples     :", len(X_test))
print("Features used    :", X_train.shape[1])

print("\n── Model Performance ───────────────────────────────")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print("────────────────────────────────────────────────────")

Training samples : 366
Test samples     : 92
Features used    : 21

── Model Performance ───────────────────────────────
MAE  : 0.9365
RMSE : 1.1817
R²   : -0.0471
────────────────────────────────────────────────────


Experiment 2 - Judge data after the point Off Spec 32 is fixed 

Result

Evidence is weak.

Before period → still poor.
After period → looks better, but sample size is too small to trust.
Conclusion

We cannot attribute the poor baseline primarily to this sensor regime change.

In [5]:
split_index = 419   # <-- Replace with your actual index

before_df = df_clean.iloc[:split_index].copy()
after_df  = df_clean.iloc[split_index:].copy()

print("Before:", before_df.shape)
print("After :", after_df.shape)

Before: (419, 24)
After : (39, 24)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_dataset(df, name):

    X = df.drop(columns=["DATE", "Wthr"])
    y = df["Wthr"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("\n", "="*50)
    print(name)
    print("="*50)
    print("Training samples :", len(X_train))
    print("Test samples     :", len(X_test))
    print("Features used    :", X_train.shape[1])

    print(f"\nMAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

evaluate_dataset(before_df, "Before Sensor Change")

evaluate_dataset(after_df, "After Sensor Change")


Before Sensor Change
Training samples : 335
Test samples     : 84
Features used    : 22

MAE  : 0.9640
RMSE : 1.2170
R²   : -0.1395

After Sensor Change
Training samples : 31
Test samples     : 8
Features used    : 22

MAE  : 0.8334
RMSE : 0.9670
R²   : 0.3461


Experiment 3 Structure
Hypothesis

There exists a process delay between sensor readings and Wthr.

In [7]:
df_lag1 = df_clean.copy()
feature_cols = df_lag1.columns.drop(["DATE", "Wthr"])

In [8]:
df_lag1[feature_cols] = df_lag1[feature_cols].shift(1)

In [9]:
df_lag1 = df_lag1.dropna().reset_index(drop=True)

In [10]:
# ── Baseline RandomForestRegressor ──────────────────────────────────────────

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Define features and target
X = df_lag1.drop(columns=["DATE", "Wthr"])
y = df_lag1["Wthr"]

# 2. Split into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Features used    : {X_train.shape[1]}\n")

# 3. Train the model
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# 4. Make predictions
y_pred = model.predict(X_test)

# 5. Evaluate
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("── Model Performance ───────────────────────────────")
print(f"  MAE  (Mean Absolute Error)       : {mae:.4f}")
print(f"  RMSE (Root Mean Squared Error)   : {rmse:.4f}")
print(f"  R²   (Coefficient of Det.)       : {r2:.4f}")
print("────────────────────────────────────────────────────\n")

# 6. First 10 actual vs predicted
comparison = pd.DataFrame({
    'Actual'   : y_test.values[:10],
    'Predicted': y_pred[:10].round(4),
    'Error'    : (y_test.values[:10] - y_pred[:10]).round(4)
})
print("── First 10 Actual vs Predicted ────────────────────")
print(comparison.to_string(index=False))
print()

# 7. Top 10 feature importances
importances = pd.Series(model.feature_importances_, index=X.columns)
top10 = importances.sort_values(ascending=False).head(10)

print("── Top 10 Feature Importances ──────────────────────")
for feat, score in top10.items():
    bar = '█' * int(score * 100)
    print(f"  {feat:<25} {score:.4f}  {bar}")
print("────────────────────────────────────────────────────")

Training samples : 365
Test samples     : 92
Features used    : 22

── Model Performance ───────────────────────────────
  MAE  (Mean Absolute Error)       : 0.9383
  RMSE (Root Mean Squared Error)   : 1.2612
  R²   (Coefficient of Det.)       : 0.0041
────────────────────────────────────────────────────

── First 10 Actual vs Predicted ────────────────────
 Actual  Predicted  Error
   -2.0     -1.380 -0.620
   -1.0     -0.686 -0.314
   -2.0     -1.115 -0.885
   -1.0     -0.898 -0.102
   -2.0     -0.928 -1.072
   -1.0     -1.360  0.360
   -2.0     -0.694 -1.306
   -1.0     -0.606 -0.394
    1.1     -0.479  1.579
   -1.0     -0.860 -0.140

── Top 10 Feature Importances ──────────────────────
  LPG Flow                  0.0963  █████████
  Stabiliser bottom level   0.0921  █████████
  Stabilizer Reflux Flow    0.0835  ████████
  Stabilizer Reflux Drum T  0.0682  ██████
  Stabilizer Feed T         0.0601  ██████
  HGO CR Flow               0.0561  █████
  Stabilized Naphtha Flow   0.0493 

In [11]:
df_clean["DATE"].diff().value_counts().head(15)

DATE
1 days    282
2 days    104
3 days     53
4 days     12
5 days      4
6 days      2
Name: count, dtype: int64

# Experiment 4 — Physics-Based Feature Selection

## Hypothesis

The Random Forest model may be learning from noisy or weakly relevant
features. Restricting the model to sensors that are physically closest to the
stabilization process may improve prediction performance.

Unlike statistical feature selection, this experiment is based on process
understanding of the Naphtha Stabilizer Column.

---

## Experimental Design

Train the same RandomForestRegressor using progressively larger groups of
features.

### Model A
Phase 1 features only

### Model B
Phase 1 + Phase 2 features

### Model C
Phase 1 + Phase 2 + Phase 3 features

### Model D
All available features (Baseline)

The model parameters, train-test split and evaluation metrics remain unchanged.
Only the input feature set changes.

---

## Feature Groups

### Phase 1 (Strong Direct Influence)

- Stabilizer Bottom Temperature
- Stabilizer Bottom Pressure
- Bottom Naphtha Reboiler Outlet Temp (TI-1908/09)
- HGO CR To Reboiler Flow
- Stabilized Naphtha Flow
- LPG Flow

---

### Phase 2 (Strong Indirect Influence)

- HGO CR Reboiler Inlet Temp (TI-1914)
- Bottom Reboiler Naphtha Inlet Temp (TI-1907)
- HGO CR Flow
- Stabilizer Bottom Level

---

### Phase 3 (Moderate Influence)

- Stabilizer Reflux Flow
- Stabilizer Top Temperature
- Stabilizer Top Pressure
- Stabilizer 3rd Tray
- Stabilizer Reflux Drum Temperature

---

### Phase 4 (Weak / Conditional)

- Stabilizer Feed Temperature
- Stabilizer Feed Flow
- Off Spec LPG from CRU Inlet Pressure

---

## Evaluation Metrics

- MAE
- RMSE
- R² Score

---

## Objective

Determine whether physically informed feature selection improves Weathering
Temperature prediction over using all available process variables.

In [17]:
# ==========================================================
# EXPERIMENT 4
# Physics-Based Feature Selection
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# -----------------------------
# Target
# -----------------------------
TARGET = "Wthr"

# -----------------------------
# Phase 1 Features
# -----------------------------
phase1 = [
    "Stabilizer Bottom T",
    "Stabilier bottom pressure",
    "Bot. Naphtha Reboiler Outlet Temp(TI-1908 & TI-1909)",
    "HGO CR to reboiler Flow",
    "Stabilized Naphtha Flow",
    "LPG Flow"
]

# -----------------------------
# Phase 2 Features
# -----------------------------
phase2 = [
    "HGO CR Reboiler Inlet Temp( TI-1914)",
    "Bottom Reboiler Naphtha Inlet Temp( TI-1907)",
    "HGO CR Flow",
    "Stabiliser bottom level"
]

# -----------------------------
# Phase 3 Features
# -----------------------------
phase3 = [
    "Stabilizer Reflux Flow",
    "Stabilizer Top T",
    "Stabilizer Top P",
    "Stab. 3rd Tray",
    "Stabilizer Reflux Drum T"
]

# -----------------------------
# Get All Features
# -----------------------------
all_features = [c for c in df_clean.columns if c != TARGET]

# -----------------------------
# Feature Sets
# -----------------------------
feature_sets = {
    "Phase 1": phase1,
    "Phase 1 + 2": phase1 + phase2,
    "Phase 1 + 2 + 3": phase1 + phase2 + phase3,
    "All Features": all_features
}

# -----------------------------
# Remove DATE if present
# -----------------------------
for key in feature_sets:
    feature_sets[key] = [
        f for f in feature_sets[key]
        if f not in ["DATE", "Date"]
    ]

# -----------------------------
# Train & Evaluate
# -----------------------------
results = []

for name, features in feature_sets.items():

    print("=" * 70)
    print(f"MODEL : {name}")
    print("=" * 70)

    X = df_clean[features]
    y = df_clean[TARGET]

    print(f"Features Used    : {len(features)}")
    print(f"Total Samples    : {len(df_clean)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    print(f"Training Samples : {len(X_train)}")
    print(f"Testing Samples  : {len(X_test)}")

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    print("\nTraining Random Forest...")
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print("\nPerformance")
    print("-" * 35)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    results.append({
        "Model": name,
        "Features": len(features),
        "Training Samples": len(X_train),
        "Testing Samples": len(X_test),
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R²": round(r2, 4)
    })

print("\n")
print("=" * 70)
print("FINAL COMPARISON")
print("=" * 70)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("R²", ascending=False).reset_index(drop=True)

display(results_df)

MODEL : Phase 1
Features Used    : 6
Total Samples    : 458
Training Samples : 366
Testing Samples  : 92

Training Random Forest...

Performance
-----------------------------------
MAE  : 0.8715
RMSE : 1.1075
R²   : 0.0802
MODEL : Phase 1 + 2
Features Used    : 10
Total Samples    : 458
Training Samples : 366
Testing Samples  : 92

Training Random Forest...

Performance
-----------------------------------
MAE  : 0.8644
RMSE : 1.0901
R²   : 0.1089
MODEL : Phase 1 + 2 + 3
Features Used    : 15
Total Samples    : 458
Training Samples : 366
Testing Samples  : 92

Training Random Forest...

Performance
-----------------------------------
MAE  : 0.9373
RMSE : 1.1749
R²   : -0.0352
MODEL : All Features
Features Used    : 22
Total Samples    : 458
Training Samples : 366
Testing Samples  : 92

Training Random Forest...

Performance
-----------------------------------
MAE  : 0.9190
RMSE : 1.1618
R²   : -0.0122


FINAL COMPARISON


,Model,Features,Training Samples,Testing Samples,MAE,RMSE,R²
0,Phase 1 + 2,10,366,92,0.8644,1.0901,0.1089
1,Phase 1,6,366,92,0.8715,1.1075,0.0802
2,All Features,22,366,92,0.9190,1.1618,-0.0122
3,Phase 1 + 2 + 3,15,366,92,0.9373,1.1749,-0.0352


# Experiment 5 — Physics-Based Feature Engineering

## Hypothesis

The increase in bottom naphtha temperature across the reboiler better
represents heat transferred to the process than individual temperature
measurements.

A higher temperature rise should indicate stronger stripping of light
hydrocarbons, which may improve Weathering Temperature prediction.

---

## Engineered Feature

Reboiler Temperature Rise

Formula

Bottom Reboiler Outlet Temperature
-
Bottom Reboiler Inlet Temperature

---

## Baseline

Model:
Random Forest

Features:
Phase 1 + Phase 2

Performance

MAE = 0.8644

RMSE = 1.0901

R² = 0.1089

---

## Success Criterion

The engineered feature is accepted only if model performance improves over the
baseline.

Conclusion: Restricting the model to the top two physics-based feature groups improved performance over the baseline. Including additional lower-ranked process variables reduced predictive performance, suggesting they introduce noise or redundant information for this model.

In [21]:
# ======================================
# Experiment 5
# Feature Engineering
# ======================================

df_exp = df_clean.copy()

df_exp["Reboiler Temperature Rise"] = (
    df_exp["Bot. Naphtha Reboiler Outlet Temp(TI-1908 & TI-1909)"]
    -
    df_exp["Bottom Reboiler Naphtha Inlet Temp( TI-1907)"]
)

In [22]:
features = phase1 + phase2

features.append("Reboiler Temperature Rise")

In [23]:
# ======================================================
# Experiment 5
# Train Random Forest with Engineered Feature
# ======================================================

# Features = Best Baseline (Phase 1 + 2) + Engineered Feature
features = phase1 + phase2 + ["Reboiler Temperature Rise"]

X = df_exp[features]
y = df_exp[TARGET]

print("=" * 70)
print("MODEL : Phase 1 + 2 + Reboiler Temperature Rise")
print("=" * 70)

print(f"Features Used    : {len(features)}")
print(f"Total Samples    : {len(df_exp)}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"Training Samples : {len(X_train)}")
print(f"Testing Samples  : {len(X_test)}")

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

print("\nTraining Random Forest...")
model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\nPerformance")
print("-" * 35)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print("\n")
print("=" * 70)
print("BASELINE COMPARISON")
print("=" * 70)

print("Previous Best (Phase 1 + 2)")
print("MAE  : 0.8644")
print("RMSE : 1.0901")
print("R²   : 0.1089")

print("\nCurrent Model")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

if r2 > 0.1089:
    print("\n✅ Hypothesis Accepted - Engineered feature improved the model.")
else:
    print("\n❌ Hypothesis Rejected - Engineered feature did not improve the model.")

MODEL : Phase 1 + 2 + Reboiler Temperature Rise
Features Used    : 11
Total Samples    : 458
Training Samples : 366
Testing Samples  : 92

Training Random Forest...

Performance
-----------------------------------
MAE  : 0.8740
RMSE : 1.0866
R²   : 0.1146


BASELINE COMPARISON
Previous Best (Phase 1 + 2)
MAE  : 0.8644
RMSE : 1.0901
R²   : 0.1089

Current Model
MAE  : 0.8740
RMSE : 1.0866
R²   : 0.1146

✅ Hypothesis Accepted - Engineered feature improved the model.


# Experiment 6 — Specific Reboiler Duty

## Hypothesis

The thermal energy delivered by the reboiler per unit of stabilized naphtha
better represents stripping efficiency than the raw process variables.

Since Weathering Temperature depends on the removal of light hydrocarbons,
a proxy for specific reboiler duty should improve prediction performance.

---

## Engineered Feature

Specific Reboiler Duty

Formula

(HGO CR Flow × Reboiler Temperature Rise)

/

Stabilized Naphtha Flow

---

## Baseline

Random Forest

Phase 1 + Phase 2

+

Reboiler Temperature Rise

R² = 0.1146

---

## Success Criterion

The feature is accepted only if it improves R² beyond 0.1146.

In [24]:
# ======================================
# Experiment 6
# Feature Engineering
# ======================================

df_exp = df_clean.copy()

# Experiment 5 feature
df_exp["Reboiler Temperature Rise"] = (
    df_exp["Bot. Naphtha Reboiler Outlet Temp(TI-1908 & TI-1909)"]
    -
    df_exp["Bottom Reboiler Naphtha Inlet Temp( TI-1907)"]
)

# Experiment 6 feature
df_exp["Specific Reboiler Duty"] = (
    df_exp["HGO CR Flow"]
    *
    df_exp["Reboiler Temperature Rise"]
) / (
    df_exp["Stabilized Naphtha Flow"] + 1e-6
)

In [25]:
features = phase1 + phase2 + [
    "Reboiler Temperature Rise",
    "Specific Reboiler Duty"
]

In [26]:
# ======================================================
# Experiment 6
# Specific Reboiler Duty
# ======================================================

# Features = Best Baseline + Accepted Engineered Feature + New Feature
features = phase1 + phase2 + [
    "Reboiler Temperature Rise",
    "Specific Reboiler Duty"
]

X = df_exp[features]
y = df_exp[TARGET]

print("=" * 70)
print("MODEL : Phase 1 + 2 + Reboiler Temperature Rise + Specific Reboiler Duty")
print("=" * 70)

print(f"Features Used    : {len(features)}")
print(f"Total Samples    : {len(df_exp)}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"Training Samples : {len(X_train)}")
print(f"Testing Samples  : {len(X_test)}")

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

print("\nTraining Random Forest...")
model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\nPerformance")
print("-" * 35)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print("\n")
print("=" * 70)
print("BASELINE COMPARISON")
print("=" * 70)

print("Previous Best")
print("Model : Phase 1 + Phase 2 + Reboiler Temperature Rise")
print("MAE   : 0.8740")
print("RMSE  : 1.0866")
print("R²    : 0.1146")

print("\nCurrent Model")
print(f"MAE   : {mae:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"R²    : {r2:.4f}")

print("\n")
if r2 > 0.1146:
    print("✅ Hypothesis Accepted")
    print("Specific Reboiler Duty improved prediction.")
else:
    print("❌ Hypothesis Rejected")
    print("Specific Reboiler Duty did not improve prediction.")

MODEL : Phase 1 + 2 + Reboiler Temperature Rise + Specific Reboiler Duty
Features Used    : 12
Total Samples    : 458
Training Samples : 366
Testing Samples  : 92

Training Random Forest...

Performance
-----------------------------------
MAE  : 0.8687
RMSE : 1.0767
R²   : 0.1308


BASELINE COMPARISON
Previous Best
Model : Phase 1 + Phase 2 + Reboiler Temperature Rise
MAE   : 0.8740
RMSE  : 1.0866
R²    : 0.1146

Current Model
MAE   : 0.8687
RMSE  : 1.0767
R²    : 0.1308


✅ Hypothesis Accepted
Specific Reboiler Duty improved prediction.


# Experiment 8 — Stripping Factor Proxy

## Hypothesis

The ratio of Bottom Temperature to Bottom Pressure better represents the
vapor-liquid equilibrium governing LPG stripping than the individual
temperature and pressure measurements.

Since Weathering Temperature depends on how easily light hydrocarbons leave
the liquid phase, this engineered feature may improve prediction.

---

## Engineered Feature

Stripping Factor Proxy

Formula

Bottom Temperature

------------------

Bottom Pressure

---

## Baseline

Current Best Model

Phase 1 + Phase 2

+

Reboiler Temperature Rise

+

Specific Reboiler Duty

R² = 0.1308

---

## Success Criterion

Accept only if performance improves beyond the current baseline.

In [29]:
# ======================================
# Experiment 8
# Feature Engineering
# ======================================

df_exp["Stripping Factor Proxy"] = (
    df_exp["Stabilizer Bottom T"]
    /
    (
        df_exp["Stabilier bottom pressure"]
        + 1e-6
    )
)

In [30]:
# ======================================================
# Experiment 8
# Stripping Factor Proxy
# ======================================================

features = phase1 + phase2 + [
    "Reboiler Temperature Rise",
    "Specific Reboiler Duty",
    "Stripping Factor Proxy"
]

X = df_exp[features]
y = df_exp[TARGET]

print("=" * 70)
print("MODEL : Phase 1 + 2 + Stripping Factor Proxy")
print("=" * 70)

print(f"Features Used    : {len(features)}")
print(f"Total Samples    : {len(df_exp)}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"Training Samples : {len(X_train)}")
print(f"Testing Samples  : {len(X_test)}")

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

print("\nTraining Random Forest...")
model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\nPerformance")
print("-" * 35)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print("\n")
print("=" * 70)
print("BASELINE COMPARISON")
print("=" * 70)

print("Previous Best")
print("MAE   : 0.8687")
print("RMSE  : 1.0767")
print("R²    : 0.1308")

print("\nCurrent Model")
print(f"MAE   : {mae:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"R²    : {r2:.4f}")

print()

if r2 > 0.1308:
    print("✅ Hypothesis Accepted")
    print("Stripping Factor Proxy improved prediction.")
else:
    print("❌ Hypothesis Rejected")
    print("Stripping Factor Proxy did not improve prediction.")

MODEL : Phase 1 + 2 + Stripping Factor Proxy
Features Used    : 13
Total Samples    : 458
Training Samples : 366
Testing Samples  : 92

Training Random Forest...

Performance
-----------------------------------
MAE  : 0.8659
RMSE : 1.0692
R²   : 0.1428


BASELINE COMPARISON
Previous Best
MAE   : 0.8687
RMSE  : 1.0767
R²    : 0.1308

Current Model
MAE   : 0.8659
RMSE  : 1.0692
R²    : 0.1428

✅ Hypothesis Accepted
Stripping Factor Proxy improved prediction.


# Experiment 9 — Algorithm Comparison

## Hypothesis

Random Forest has produced encouraging results after physics-based feature
engineering.

However, XGBoost is often better at learning complex nonlinear relationships
between engineered features.

This experiment keeps the dataset and feature set identical while changing only
the learning algorithm.

---

## Model Comparison

Random Forest

vs

XGBoost Regressor

---

## Feature Set

Phase 1

Phase 2

Reboiler Temperature Rise

Specific Reboiler Duty

Stripping Factor Proxy

---

## Baseline

Random Forest

R² = 0.1428

MAE = 0.8659

RMSE = 1.0692

---

## Success Criterion

If XGBoost achieves higher R² and lower prediction error,
it will become the final prediction model.

In [32]:
from xgboost import XGBRegressor

In [33]:
# ======================================================
# Experiment 9
# XGBoost Regressor
# ======================================================

features = phase1 + phase2 + [
    "Reboiler Temperature Rise",
    "Specific Reboiler Duty",
    "Stripping Factor Proxy"
]

X = df_exp[features]
y = df_exp[TARGET]

print("=" * 70)
print("MODEL : XGBoost Regressor")
print("=" * 70)

print(f"Features Used    : {len(features)}")
print(f"Total Samples    : {len(df_exp)}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"Training Samples : {len(X_train)}")
print(f"Testing Samples  : {len(X_test)}")

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

print("\nTraining XGBoost...")
model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\nPerformance")
print("-"*35)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print("\n")
print("=" * 70)
print("RANDOM FOREST vs XGBOOST")
print("=" * 70)

print("Random Forest")
print("MAE  : 0.8659")
print("RMSE : 1.0692")
print("R²   : 0.1428")

print("\nXGBoost")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print()

if r2 > 0.1428:
    print("✅ Hypothesis Accepted")
    print("XGBoost outperformed Random Forest.")
else:
    print("❌ Hypothesis Rejected")
    print("Random Forest remains the better model.")

MODEL : XGBoost Regressor
Features Used    : 13
Total Samples    : 458
Training Samples : 366
Testing Samples  : 92

Training XGBoost...

Performance
-----------------------------------
MAE  : 0.9136
RMSE : 1.1304
R²   : 0.0418


RANDOM FOREST vs XGBOOST
Random Forest
MAE  : 0.8659
RMSE : 1.0692
R²   : 0.1428

XGBoost
MAE  : 0.9136
RMSE : 1.1304
R²   : 0.0418

❌ Hypothesis Rejected
Random Forest remains the better model.


# Experiment 10 — Cross Validation

## Hypothesis

The best Random Forest model should produce consistent performance across
multiple train/test splits rather than relying on a single random split.

Cross-validation provides a more reliable estimate of real-world model
performance.

---

## Model

Random Forest Regressor

---

## Feature Set

Phase 1

Phase 2

Reboiler Temperature Rise

Specific Reboiler Duty

Stripping Factor Proxy

---

## Validation

5-Fold Cross Validation

Scoring

R²

MAE

RMSE

---

## Success Criterion

If the average cross-validation score remains close to the previous test score,
the model is considered stable.

In [34]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_validate

In [35]:
# ======================================================
# Experiment 10
# 5-Fold Cross Validation
# ======================================================

from sklearn.model_selection import KFold, cross_validate

features = phase1 + phase2 + [
    "Reboiler Temperature Rise",
    "Specific Reboiler Duty",
    "Stripping Factor Proxy"
]

X = df_exp[features]
y = df_exp[TARGET]

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error"
    }
)

print("="*70)
print("5-FOLD CROSS VALIDATION")
print("="*70)

print(f"Mean R²   : {scores['test_r2'].mean():.4f}")
print(f"Std  R²   : {scores['test_r2'].std():.4f}")

print()

print(f"Mean MAE  : {-scores['test_mae'].mean():.4f}")
print(f"Mean RMSE : {-scores['test_rmse'].mean():.4f}")

print("\nIndividual Fold R²")

for i, score in enumerate(scores["test_r2"], start=1):
    print(f"Fold {i}: {score:.4f}")

5-FOLD CROSS VALIDATION
Mean R²   : 0.0455
Std  R²   : 0.0929

Mean MAE  : 0.9211
Mean RMSE : 1.1896

Individual Fold R²
Fold 1: 0.1292
Fold 2: -0.1292
Fold 3: 0.0663
Fold 4: 0.0432
Fold 5: 0.1179
